# **Tahapan Analisis Sentimen Ulasan Shopee**

# 1. INSTALL

Tahap ini dilakukan untuk menginstal library yang dibutuhkan seperti google-play-scraper dan Sastrawi sebagai alat bantu dalam proses pengambilan data dan preprocessing teks.



In [74]:
!pip install google-play-scraper Sastrawi

# 2. SCRAPING DATA (SHOPEE)


Pada tahap ini dilakukan pengambilan data ulasan pengguna aplikasi Shopee dari Google Play Store. Data yang diperoleh berupa teks ulasan dan rating yang akan digunakan sebagai dataset utama dalam penelitian.

In [75]:
from google_play_scraper import reviews
import pandas as pd

result, _ = reviews(
    'com.shopee.id',
    lang='id',
    country='id',
    count=10000
)

df = pd.DataFrame(result)
df = df[['content', 'score']]

df.to_csv('dataset.csv', index=False)

print("Scraping selesai")

Scraping selesai


# 3. LABELING

Data rating diubah menjadi kategori sentimen yaitu negatif, netral, dan positif. Tahap ini bertujuan untuk memberikan label pada data sehingga dapat digunakan dalam proses klasifikasi.

In [76]:
def label_sentiment(score):
    if score <= 2:
        return 'negatif'
    elif score == 3:
        return 'netral'
    else:
        return 'positif'

df['sentiment'] = df['score'].apply(label_sentiment)

# 4. PREPROCESSING

Teks dibersihkan melalui proses seperti mengubah huruf menjadi lowercase, menghapus simbol, dan menghilangkan stopword. Tahap ini penting untuk mengurangi noise dan meningkatkan kualitas data.

In [77]:
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

factory = StopWordRemoverFactory()
stopword = factory.create_stop_word_remover()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = stopword.remove(text)
    return text

df['clean'] = df['content'].apply(clean_text)

# 5. TOKENIZING DAN PADDING


Teks yang telah dibersihkan diubah menjadi bentuk numerik menggunakan tokenizing, kemudian dilakukan padding agar panjang data seragam sehingga dapat diproses oleh model.




In [78]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(df['clean'])

X = tokenizer.texts_to_sequences(df['clean'])
X = pad_sequences(X, maxlen=100)

# 6. LABEL ENCODING

Label sentimen diubah menjadi bentuk numerik agar dapat dipahami oleh model deep learning dalam proses pelatihan.

In [79]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df['sentiment'])

# 7. SPLIT DATA


Dataset dibagi menjadi data latih dan data uji dengan tujuan untuk melatih model dan menguji performa model secara objektif.



In [80]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 8. CALLBACK

Pada tahap ini digunakan teknik early stopping untuk menghentikan proses pelatihan model secara otomatis ketika performa model pada data validasi tidak lagi mengalami peningkatan. Hal ini bertujuan untuk mencegah terjadinya overfitting, yaitu kondisi ketika model terlalu menghafal data latih sehingga performanya menurun pada data uji. Selain itu, penggunaan callback juga membantu mendapatkan bobot model terbaik selama proses pelatihan.

In [81]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# 9. MODEL 1 (LSTM)

Model deep learning yang digunakan meliputi LSTM, GRU, dan Dense. Model ini digunakan untuk mempelajari pola dalam data teks dan melakukan klasifikasi sentimen.

In [82]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model1 = Sequential([
    Embedding(10000, 64),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model1.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model1.fit(
    X_train, y_train,
    epochs=10,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 30s 84ms/step - accuracy: 0.8171 - loss: 0.5050 - val_accuracy: 0.8665 - val_loss: 0.3817
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 22s 90ms/step - accuracy: 0.8996 - loss: 0.3148 - val_accuracy: 0.8745 - val_loss: 0.3728
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 35s 67ms/step - accuracy: 0.9244 - loss: 0.2386 - val_accuracy: 0.8670 - val_loss: 0.3923
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 16s 65ms/step - accuracy: 0.9423 - loss: 0.1788 - val_accuracy: 0.8585 - val_loss: 0.4857


# 10. MODEL 2 (GRU + DROPOUT)

In [83]:
from tensorflow.keras.layers import GRU

model2 = Sequential([
    Embedding(10000, 64),
    GRU(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model2.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model2.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test), callbacks=[early_stop])

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 22s 71ms/step - accuracy: 0.8031 - loss: 0.5021 - val_accuracy: 0.8660 - val_loss: 0.3939
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 22s 75ms/step - accuracy: 0.8992 - loss: 0.3123 - val_accuracy: 0.8655 - val_loss: 0.3885


# 11. MODEL 3 (DENSE)

In [84]:
from tensorflow.keras.layers import Flatten

model3 = Sequential([
    Embedding(10000, 64),
    Flatten(),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model3.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model3.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test), callbacks=[early_stop])

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8269 - loss: 0.4673 - val_accuracy: 0.8700 - val_loss: 0.3743
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.9006 - loss: 0.3002 - val_accuracy: 0.8715 - val_loss: 0.3717
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.9385 - loss: 0.1964 - val_accuracy: 0.8610 - val_loss: 0.4304
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.9681 - loss: 0.1213 - val_accuracy: 0.8610 - val_loss: 0.4700


# 12. EVALUASI

Model diuji menggunakan data uji untuk mengetahui tingkat akurasi. Hasil menunjukkan bahwa model mampu mencapai akurasi di atas 85%.



In [85]:
print("LSTM:", model1.evaluate(X_test, y_test))
print("GRU:", model2.evaluate(X_test, y_test))
print("DENSE:", model3.evaluate(X_test, y_test))
print("Model selesai dilatih dan dievaluasi")

63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.8745 - loss: 0.3728
LSTM: [0.37282809615135193, 0.8744999766349792]
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.8660 - loss: 0.3939
GRU: [0.3939395248889923, 0.8659999966621399]
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8715 - loss: 0.3717
DENSE: [0.37171921133995056, 0.8715000152587891]
Model selesai dilatih dan dievaluasi


# 13. INFERENCE

Model digunakan untuk memprediksi sentimen dari teks baru, sehingga dapat diketahui apakah suatu ulasan termasuk positif, netral, atau negatif.

In [86]:
import numpy as np

text = ["aplikasi sangat bagus"]

seq = tokenizer.texts_to_sequences(text)
pad = pad_sequences(seq, maxlen=100)

pred = model1.predict(pad)

label = ["negatif", "netral", "positif"]
print("Hasil Prediksi:", label[np.argmax(pred)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step
Hasil Prediksi: positif


# 14. SIMPAN FILE

In [87]:
df.to_csv("dataset.csv", index=False)
!pip freeze > requirements.txt

# KESIMPULAN:

Berdasarkan hasil penelitian, model deep learning yang digunakan mampu mengklasifikasikan sentimen ulasan pengguna Shopee dengan akurasi di atas 85%. Model LSTM menunjukkan performa terbaik dibandingkan model lainnya. Penggunaan preprocessing dan teknik seperti early stopping terbukti membantu meningkatkan kestabilan model serta mencegah overfitting. Meskipun demikian, performa model masih dapat ditingkatkan karena adanya noise pada data ulasan pengguna.